In [16]:
import nflreadpy as nfl
import pandas as pd
import numpy as np

In [21]:
pbp = nfl.load_pbp([2025])
pbp = pbp.to_pandas()

In [22]:
offense_df = (
    pbp
    .groupby(['game_id', 'posteam'])
    .agg(
        offensive_epa = ('epa', 'mean'),
        success_rate = ('success', 'mean')
    )
    .reset_index()
)

In [77]:
pass_df = pbp[pbp['pass'] == 1]
rush_df = pbp[pbp['rush'] == 1]

In [78]:
defense_df = (
    pbp
    .groupby(['game_id', 'defteam'])
    .agg(
        defensive_epa_allowed = ('epa', 'mean'),
        defensive_success_rate_allowed = ('success', 'mean')
    )
    .reset_index()
)

In [79]:
schedule = nfl.load_schedules([2025])
schedule = schedule.to_pandas()

In [80]:
schedule_df = schedule[['game_id', 'home_team', 'away_team', 'home_score', 'away_score', 'week', 'season']]
schedule_df.head()

,game_id,home_team,away_team,home_score,away_score,week,season
0,2025_01_DAL_PHI,PHI,DAL,24,20,1,2025
1,2025_01_KC_LAC,LAC,KC,27,21,1,2025
2,2025_01_TB_ATL,ATL,TB,20,23,1,2025
3,2025_01_CIN_CLE,CLE,CIN,16,17,1,2025
4,2025_01_MIA_IND,IND,MIA,33,8,1,2025


In [81]:
schedule_df["home_win"] = (schedule_df["home_score"] > schedule_df["away_score"]).astype(int)
schedule_df = schedule_df[['game_id', 'home_team', 'away_team', 'home_score', 'away_score', 'home_win', 'week', 'season']]
schedule_df.head()

,game_id,home_team,away_team,home_score,away_score,home_win,week,season
0,2025_01_DAL_PHI,PHI,DAL,24,20,1,1,2025
1,2025_01_KC_LAC,LAC,KC,27,21,1,1,2025
2,2025_01_TB_ATL,ATL,TB,20,23,0,1,2025
3,2025_01_CIN_CLE,CLE,CIN,16,17,0,1,2025
4,2025_01_MIA_IND,IND,MIA,33,8,1,1,2025


In [82]:
df = offense_df.merge(defense_df, left_on=['game_id', 'posteam'], right_on=['game_id', 'defteam'], how='left')
df = df.merge(pass_df.groupby(['game_id', 'posteam']).agg(pass_epa = ('epa', 'mean')).reset_index(), on=['game_id', 'posteam'], how='left')
df = df.merge(rush_df.groupby(['game_id', 'posteam']).agg(rush_epa = ('epa', 'mean')).reset_index(), on=['game_id', 'posteam'], how='left')
df.head()

,game_id,posteam,offensive_epa,success_rate,defteam,defensive_epa_allowed,defensive_success_rate_allowed,pass_epa,rush_epa
0,2025_01_ARI_NO,ARI,0.016025,0.444444,ARI,-0.090947,0.421053,0.131530,-0.052630
1,2025_01_ARI_NO,NO,-0.090947,0.421053,NO,0.016025,0.444444,-0.014034,-0.157067
2,2025_01_BAL_BUF,BAL,0.340293,0.563380,BAL,0.203761,0.490566,0.530099,0.282997
3,2025_01_BAL_BUF,BUF,0.203761,0.490566,BUF,0.340293,0.563380,0.383636,-0.046021
4,2025_01_CAR_JAX,CAR,-0.108388,0.394737,CAR,0.117530,0.481481,-0.214952,-0.092878
